# Entropy-Balanced Logit & Survival

Robustness check for `jn_r_02_IPW.ipynb`. Replaces CBPS inverse-probability weighting with
entropy balancing (Hainmueller 2012; Vegetabile et al. 2021) via
`WeightIt::weightit(..., method = "ebal", moments = 1)`. EB directly imposes exact
mean-balance of all covariates with treatment without modelling the propensity score,
making a stronger identification claim suitable for use as a robustness check.

Outputs: `Output/Tables/EB*.tex` and `Output/Images/Graphs/eb_*.png`.
Run the setup cell first, then sections in order.

In [1]:
# jn_r_05_entropy_balancing.ipynb — Entropy-balanced logit and survival models
# Robustness check for jn_r_02_IPW.ipynb. Replaces CBPS with entropy balancing.
# Sections:
#   1. Main specifications: total and split (small/large) monastic land — 6 specs
#   2. OwnOther specification: on-site vs off-site land — 3 specs
# Each section produces stargazer tables and coefficient plots.

pacman::p_load(
  sf, tidyverse, stargazer, sp, dplyr,
  cem, MatchIt, WeightIt, marginaleffects, ipw,
  survey, optmatch, conflicted, cobalt, twang,
  survival, ggplot2, broom, jsonlite
)
conflict_prefer("filter", "dplyr")
conflict_prefer("select", "dplyr")

PROJECT_ROOT <- tryCatch(
  normalizePath(file.path(dirname(rstudioapi::getActiveDocumentContext()$path), "..")),
  error = function(e) {
    cwd <- normalizePath(getwd())
    if (basename(cwd) == "Code") dirname(cwd) else cwd
  }
)
setwd(PROJECT_ROOT)
# Load pretty dictionary for labels
pretty_dict <- fromJSON("Code/pretty_dict.json")

pdf <- read_sf(dsn = "Data/Processed/northParishFlows.shp")

# Replace NAs in terrainTyp with 'Other'
pdf$terrainTyp <- ifelse(is.na(pdf$terrainTyp), "Other", pdf$terrainTyp)
pdf$uplands    <- ifelse(pdf$terrainTyp == "Uplands",  1, 0)
pdf$lowlands   <- ifelse(pdf$terrainTyp == "Lowlands", 1, 0)
pdf$otherlands <- ifelse(pdf$terrainTyp == "Other",    1, 0)

rdf <- data.frame(pdf)
day <- 40
rdf$day <- replace(rdf$day, rdf$day < 1, day)
rdf$day <- ifelse(is.na(rdf$day), day, rdf$day)
rdf$primary_day <- rdf$day * rdf$primary
rdf$primary_day <- replace(rdf$primary_day, rdf$primary_day < 1, day)
rdf$survival <- rdf$day - rdf$news_day
rdf$primary_survival <- rdf$primary_day - rdf$news_day
rdf$primary_survival <- ifelse(is.na(rdf$primary_survival), day, rdf$primary_survival)

# Convert seats to binary (1 if seats > 1, 0 otherwise)
rdf$seats <- ifelse(rdf$seats > 1, 1, rdf$seats)

# Standardize and center continuous variables.
# Binary dummies (smHouse, bigHouse, mg_fsnub, mg_court, friary) are NOT standardized.
for (v in c(
  # Total monastic land (3 normalizations)
  "llandOwned", "llo_sk", "llo_arak",
  # Small/large split land (3 normalizations)
  "lsmLand",   "lbigLand",
  "lsm_sk",    "lbg_sk",
  "lsm_arak",  "lbg_arak",
  # Off-site/on-site split land (3 normalizations)
  "lotherLand", "lownLand",
  "loth_sk",    "lown_sk",
  "loth_arak",  "lown_arak",
  # Tithes, alms, net income (3 normalizations each)
  "ltitheOutT", "lti_sk", "lti_arak",
  "lalmsInTot", "lal_sk", "lal_arak",
                "lni_sk", "lni_arak",
  # Controls (continuous)
  "lLStax_pc", "lpopC", "distScot", "area", "mean_slope",
  "wet_1535", "wet_1536"
)) {
  rdf[[v]] <- scale(rdf[[v]], center = TRUE, scale = TRUE)[, 1]
}

# --- Shared "after monastic" covariates common to every main spec ---------
# smHouse / bigHouse / mg_fsnub / mg_court: binary 20km proximity dummies — not standardized.
main_shared_after <- c("smHouse", "bigHouse", "friary", "mg_fsnub", "mg_court")

# --- Shared geographic / socioeconomic controls ---------------------------
covar_rhs <- c(
  "lLStax_pc", "lpopC", "distScot", "area",
  "uplands", "lowlands", "mean_slope", "wet_1535", "wet_1536"
)

# --- Six main specifications (total and split, 3 normalizations each) -----
# treatment = main explanatory variable (large house or total land)
# mon_covars = complementary monastic variables in the outcome model
main_specs <- list(
  total_raw = list(
    treat      = "llandOwned",
    mon_covars = c("ltitheOutT", "lalmsInTot"),
    suffix     = "_total_raw"
  ),
  total_sk = list(
    treat      = "llo_sk",
    mon_covars = c("lti_sk", "lal_sk"),
    suffix     = "_total_sk"
  ),
  total_arak = list(
    treat      = "llo_arak",
    mon_covars = c("lti_arak", "lal_arak", "lni_arak"),
    suffix     = "_total_arak"
  ),
  split_raw = list(
    treat      = "lbigLand",
    mon_covars = c("lsmLand", "ltitheOutT", "lalmsInTot"),
    suffix     = "_split_raw"
  ),
  split_sk = list(
    treat      = "lbg_sk",
    mon_covars = c("lsm_sk", "lti_sk", "lal_sk"),
    suffix     = "_split_sk"
  ),
  split_arak = list(
    treat      = "lbg_arak",
    mon_covars = c("lsm_arak", "lti_arak", "lal_arak", "lni_arak"),
    suffix     = "_split_arak"
  )
)

[conflicted] Will prefer dplyr::filter over any other package.


[conflicted] Will prefer dplyr::select over any other package.


## Helper: coefficient extraction and plotting

In [2]:
extract_coefs_svyglm <- function(model, var_name) {
  coef_summary <- summary(model)$coefficients
  coef_val <- coef_summary[var_name, "Estimate"]
  se_val   <- coef_summary[var_name, "Std. Error"]
  z_crit   <- qnorm(0.95)  # 90% CI
  ci_lower <- coef_val - z_crit * se_val
  ci_upper <- coef_val + z_crit * se_val
  z_stat   <- coef_val / se_val
  p_val    <- 2 * pnorm(abs(z_stat), lower.tail = FALSE)
  data.frame(variable = var_name, coefficient = coef_val, se = se_val,
             ci_lower = ci_lower, ci_upper = ci_upper, p_value = p_val)
}

extract_coefs_coxph <- function(model, var_name) {
  coef_summary <- summary(model)$coefficients
  coef_val <- coef_summary[var_name, "coef"]
  se_val   <- coef_summary[var_name, "se(coef)"]
  z_crit   <- qnorm(0.95)
  ci_lower <- coef_val - z_crit * se_val
  ci_upper <- coef_val + z_crit * se_val
  z_stat   <- coef_val / se_val
  p_val    <- 2 * pnorm(abs(z_stat), lower.tail = FALSE)
  data.frame(variable = var_name, coefficient = coef_val, se = se_val,
             ci_lower = ci_lower, ci_upper = ci_upper, p_value = p_val)
}

make_coef_df_ipw <- function(model, vars, extract_fn) {
  coefs <- bind_rows(lapply(vars, function(v) extract_fn(model, v)))
  coefs$significant <- ifelse(is.na(coefs$p_value), FALSE, coefs$p_value < 0.10)
  coefs$order       <- match(coefs$variable, vars)
  coefs
}

make_ipw_plot <- function(coef_df, var_labels, x_label = "Coefficient (Log Odds)") {
  # Look up pretty labels; unlist first so missing keys yield NA (preserving length)
  labels_vec <- unlist(var_labels)
  coef_df$variable_label <- unname(labels_vec[coef_df$variable])
  # Pre-compute ordered factor outside aes() to avoid IRkernel tidy-eval quirk
  lvl_order <- order(-coef_df$order)
  coef_df$variable_label <- factor(coef_df$variable_label,
                                    levels = coef_df$variable_label[lvl_order])
  ggplot(coef_df, aes(x = coefficient, y = variable_label)) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "gray50") +
    geom_errorbar(aes(xmin = ci_lower, xmax = ci_upper),
                  width = 0.2, color = "gray30", orientation = "y") +
    geom_point(aes(color = significant), size = 3) +
    scale_color_manual(
      values = c("FALSE" = "gray60", "TRUE" = "#0072B2"),
      labels = c("FALSE" = "Not Significant", "TRUE" = "p < 0.10")
    ) +
    labs(x = x_label, y = "", color = "Significance") +
    theme_minimal() +
    theme(
      axis.text.x  = element_text(size = 16),
      axis.text.y  = element_text(size = 16),
      axis.title.x = element_text(size = 16),
      legend.text  = element_text(size = 15),
      legend.title = element_text(size = 15),
      legend.position = "bottom"
    )
}

In [3]:
# --- Weighted Conley (spatial HAC) sandwich for EB svyglm logit -----------
# conleyreg does not accept observation weights, so we implement the weighted
# spatial-HAC sandwich directly. Returns a named numeric vector of SEs
# aligned to coef(svy_mod).

parish_xy_eb <- sf::st_coordinates(sf::st_centroid(sf::st_geometry(pdf)))
rdf$.cx <- parish_xy_eb[, 1]
rdf$.cy <- parish_xy_eb[, 2]

weighted_conley_se <- function(svy_mod, wts_full, cutoff_km = 100,
                               kernel = "bartlett") {
  mf <- model.frame(svy_mod)
  X  <- model.matrix(svy_mod)
  y  <- model.response(mf)
  mu <- as.numeric(fitted(svy_mod))
  used_rows <- as.integer(rownames(mf))
  w  <- wts_full[used_rows]
  cx <- rdf$.cx[used_rows]; cy <- rdf$.cy[used_rows]

  fam <- svy_mod$family$family
  if (grepl("binomial", fam)) {
    Avar <- mu * (1 - mu)
  } else if (grepl("poisson", fam)) {
    Avar <- mu
  } else {
    Avar <- rep(1, length(mu))
  }

  WX_bread <- X * (w * Avar)
  bread    <- solve(crossprod(X, WX_bread))
  u  <- as.numeric(w * (y - mu))
  Xu <- X * u

  d_km <- as.matrix(stats::dist(cbind(cx, cy))) / 1000
  K    <- pmax(1 - d_km / cutoff_km, 0)   # Bartlett kernel

  meat <- crossprod(Xu, K %*% Xu)
  V    <- bread %*% meat %*% bread
  setNames(sqrt(diag(V)), colnames(X))
}

## Section 1: Main specification (entropy-balanced)

In [4]:
# Fit entropy-balanced logit and Cox PH models for every main spec (6 total).
# Results stored in main_results for use in the Conley section.
main_results <- list()

for (spec_name in names(main_specs)) {
  spec   <- main_specs[[spec_name]]
  sfx    <- spec$suffix
  tr     <- spec$treat
  monc   <- spec$mon_covars
  covars <- c(monc, main_shared_after, covar_rhs)
  rhs    <- paste(c(tr, covars), collapse = " + ")

  cat(sprintf("\n===== EB spec [%s] — treatment %s =====\n", spec_name, tr))

  # Entropy balancing: directly imposes mean-balance (moments = 1) of all
  # covariates with the continuous treatment, no propensity score modelled.
  wt <- weightit(
    as.formula(paste(tr, "~", paste(covars, collapse = " + "))),
    data = rdf, method = "ebal", moments = 1
  )
  wts <- wt$weights
  # ESS for continuous-treatment EB (wt$ESS not populated by WeightIt for ebal)
  ess <- sum(wts)^2 / sum(wts^2)
  cat("  ESS:", round(ess, 1), "| n:", nrow(rdf), "\n")
  print(summary(wt))
  design <- svydesign(~1, weights = wts, data = rdf)

  # maxit = 200 avoids non-convergence in sparse binary outcomes (e.g. total_raw)
  wlm_primary <- svyglm(as.formula(paste("primary ~", rhs)),
                        data = rdf, weights = wts, design = design,
                        family = quasibinomial(),
                        control = glm.control(maxit = 200))
  wlm_muster  <- svyglm(as.formula(paste("muster ~",  rhs)),
                        data = rdf, weights = wts, design = design,
                        family = quasibinomial(),
                        control = glm.control(maxit = 200))
  wlm_seats   <- svyglm(as.formula(paste("seats ~",   rhs)),
                        data = rdf, weights = wts, design = design,
                        family = quasibinomial(),
                        control = glm.control(maxit = 200))
  wsurv <- coxph(as.formula(paste("Surv(primary_survival, primary) ~", rhs)),
                 data = rdf, weights = wts, robust = TRUE)

  main_results[[spec_name]] <- list(
    muster = wlm_muster, primary = wlm_primary,
    seats  = wlm_seats,  surv    = wsurv,
    wts = wts, tr = tr, covars = covars, monc = monc
  )

  # Back-compat aliases (suffixed _eb to avoid collision with IPW notebook globals)
  if (spec_name == "split_arak") {
    wt_eb            <<- wt
    weights_eb       <<- wts
    design_eb        <<- design
    wlm_primary_eb   <<- wlm_primary
    wlm_muster_eb    <<- wlm_muster
    wlm_seats_eb     <<- wlm_seats
    wsurv_eb         <<- wsurv
  }

  print(summary(wlm_primary))
  print(summary(wsurv))

  # Covariate label order: treatment → complementary monastic vars → dummies → controls
  eb_cov_order  <- c(tr, monc, main_shared_after,
                     "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")
  eb_cov_labels <- unlist(pretty_dict[eb_cov_order])

  # Conley SEs (100km Bartlett) for the three svyglm logit columns.
  # coxph with robust = TRUE already carries Lin-Wei robust SEs.
  se_muster  <- weighted_conley_se(wlm_muster,  wts)
  se_primary <- weighted_conley_se(wlm_primary, wts)
  se_seats   <- weighted_conley_se(wlm_seats,   wts)

  # capture.output + writeLines avoids the stargazer out= bug with survey objects
  tex_lines <- capture.output(
    stargazer(wlm_muster, wlm_primary, wlm_seats, wsurv,
      type = "latex",
      title = paste0("EB Logit and Cox PH Models — Muster, Primary, Seats [", spec_name, "]"),
      label = paste0("tab:eb", sfx),
      align = TRUE,
      table.placement = "H",
      column.labels = c("Muster", "Primary", "Seats", "Cox PH"),
      se = list(se_muster, se_primary, se_seats, NULL),
      order = paste0("^", eb_cov_order, "$"),
      covariate.labels = eb_cov_labels,
      omit = c("Constant", "uplands", "lowlands", "area", "mean_slope"),
      add.lines = list(
        c("Geographic Controls",  "Y", "Y", "Y", "Y"),
        c("EB weights",           "Y", "Y", "Y", "Y"),
        c("Conley SEs (100 km)",  "Y", "Y", "Y", "—"),
        c("Robust SEs (Lin-Wei)", "—", "—", "—", "Y")
      ),
      column.sep.width = ".5pt",
      omit.stat = c("aic", "lr", "wald", "logrank")
    )
  )
  writeLines(tex_lines, paste0("Output/Tables/EB", sfx, ".tex"))

  vars_to_plot <- c(tr, monc, main_shared_after,
                    "lLStax_pc", "wet_1535", "wet_1536", "lpopC")

  ggsave(paste0("Output/Images/Graphs/eb_logit_primary_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_primary, vars_to_plot, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/eb_logit_muster_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_muster, vars_to_plot, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/eb_logit_seats_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_seats, vars_to_plot, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/eb_cox_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wsurv, vars_to_plot, extract_coefs_coxph),
           pretty_dict, x_label = "Coefficient (Log Hazard Ratio)"),
         width = 10, height = 6, dpi = 300)
}
cat("Section 1 complete — 6 specs.\n")


===== EB spec [total_raw] — treatment llandOwned =====


Warning message:
"Missing values are present in the covariates. See
`?WeightIt::method_ebal` for information on how these are handled."


  ESS: 1062.5 | n: 1755 
                  Summary of weights

- Weight ranges:

      Min                                  Max
all 0.004 |---------------------------| 16.107

- Units with the 5 most extreme weights:
                                   
      1508  1278  1212   546     97
 all 6.425 6.845 8.087 8.793 16.107

- Weight statistics:

    Coef of Var   MAD Entropy # Zeros
all       0.808 0.438   0.211       0

- Effective Sample Sizes:

             Total
Unweighted 1755.  
Weighted   1062.53


Warning message:
"glm.fit: algorithm did not converge"



Call:
svyglm(formula = as.formula(paste("primary ~", rhs)), design = design, 
    family = quasibinomial(), data = rdf, weights = wts, control = glm.control(maxit = 200))

Survey design:
svydesign(~1, weights = wts, data = rdf)

Coefficients:
             Estimate Std. Error t value Pr(>|t|)  
(Intercept) -3.307303   1.825654  -1.812   0.0703 .
llandOwned   0.584716   0.411477   1.421   0.1555  
ltitheOutT  -0.395489   0.462668  -0.855   0.3928  
lalmsInTot  -0.637229   0.317422  -2.008   0.0449 *
smHouse     -2.167103   0.897229  -2.415   0.0159 *
bigHouse     0.838991   0.692737   1.211   0.2261  
friary      -0.128109   1.037650  -0.123   0.9018  
mg_fsnub    -0.941587   0.977614  -0.963   0.3356  
mg_court    -0.032547   0.661774  -0.049   0.9608  
lLStax_pc    0.148682   0.410959   0.362   0.7176  
lpopC        0.305555   0.223102   1.370   0.1710  
distScot    -0.807527   0.444192  -1.818   0.0693 .
area        -0.177017   0.357945  -0.495   0.6210  
uplands     -0.009287   1.31

...

  ESS: 1267.3 | n: 1755 
                  Summary of weights

- Weight ranges:

      Min                                 Max
all 0.001 |---------------------------| 5.841

- Units with the 5 most extreme weights:
                                 
     1415   546   528   119    97
 all 4.67 5.101 5.226 5.688 5.841

- Weight statistics:

    Coef of Var   MAD Entropy # Zeros
all       0.621 0.401   0.156       0

- Effective Sample Sizes:

             Total
Unweighted 1755.  
Weighted   1267.27

Call:
svyglm(formula = as.formula(paste("primary ~", rhs)), design = design, 
    family = quasibinomial(), data = rdf, weights = wts, control = glm.control(maxit = 200))

Survey design:
svydesign(~1, weights = wts, data = rdf)

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept) -4.793663   1.341411  -3.574 0.000364 ***
lbg_sk       0.754425   0.398166   1.895 0.058336 .  
lsm_sk      -0.689655   0.288074  -2.394 0.016798 *  
lti_sk      -0.212497   0.290936  -0.73


===== EB spec [split_arak] — treatment lbg_arak =====


Warning message:
"Missing values are present in the covariates. See
`?WeightIt::method_ebal` for information on how these are handled."


  ESS: 1177.4 | n: 1755 
                  Summary of weights

- Weight ranges:

    Min                                 Max
all   0 |---------------------------| 9.603

- Units with the 5 most extreme weights:
                                  
      1415  1402  1160  1089    97
 all 6.544 6.665 6.702 7.758 9.603

- Weight statistics:

    Coef of Var   MAD Entropy # Zeros
all       0.701 0.416   0.177       0

- Effective Sample Sizes:

             Total
Unweighted 1755.  
Weighted   1177.41

Call:
svyglm(formula = as.formula(paste("primary ~", rhs)), design = design, 
    family = quasibinomial(), data = rdf, weights = wts, control = glm.control(maxit = 200))

Survey design:
svydesign(~1, weights = wts, data = rdf)

Coefficients:
             Estimate Std. Error t value Pr(>|t|)  
(Intercept) -4.364595   1.989988  -2.193   0.0285 *
lbg_arak     0.617274   0.321680   1.919   0.0552 .
lsm_arak    -0.317194   0.261983  -1.211   0.2262  
lti_arak    -0.322720   0.406363  -0.794   0.427

Section 1 complete — 6 specs.


## Section 2: OwnOther specification (entropy-balanced)

In [5]:
oo_specs <- list(
  ownOther_raw = list(
    treat        = "lotherLand",
    own_var      = "lownLand",
    other_covars = c("ltitheOutT", "lalmsInTot"),
    suffix       = "_ownOther_raw"
  ),
  ownOther_sk = list(
    treat        = "loth_sk",
    own_var      = "lown_sk",
    other_covars = c("lti_sk", "lal_sk"),
    suffix       = "_ownOther_sk"
  ),
  ownOther_arak = list(
    treat        = "loth_arak",
    own_var      = "lown_arak",
    other_covars = c("lti_arak", "lal_arak"),  # lni_arak omitted to avoid collinearity
    suffix       = "_ownOther_arak"
  )
)

oo_results <- list()

for (spec_name in names(oo_specs)) {
  spec         <- oo_specs[[spec_name]]
  sfx          <- spec$suffix
  oo_treat     <- spec$treat
  oo_own       <- spec$own_var
  other_covars <- spec$other_covars

  oo_covars         <- c(oo_own, other_covars, main_shared_after, covar_rhs)
  oo_balance_covars <- setdiff(oo_covars, oo_own)
  oo_all            <- c(oo_treat, oo_covars)
  oo_formula_rhs    <- paste(oo_all, collapse = " + ")

  cat(sprintf("\n===== OwnOther EB spec [%s] — treatment %s =====\n", spec_name, oo_treat))

  wt_oo <- weightit(
    as.formula(paste(oo_treat, "~", paste(oo_balance_covars, collapse = " + "))),
    data = rdf, method = "ebal", moments = 1
  )
  weights_oo <- wt_oo$weights
  # ESS for continuous-treatment EB (wt$ESS not populated by WeightIt for ebal)
  ess_oo <- sum(weights_oo)^2 / sum(weights_oo^2)
  cat("  ESS:", round(ess_oo, 1), "| n:", nrow(rdf), "\n")
  print(summary(wt_oo))
  design_oo <- svydesign(~1, weights = weights_oo, data = rdf)

  wlm_primary_oo <- svyglm(as.formula(paste("primary ~", oo_formula_rhs)),
                            data = rdf, weights = weights_oo, design = design_oo,
                            family = quasibinomial(),
                            control = glm.control(maxit = 200))
  wlm_muster_oo  <- svyglm(as.formula(paste("muster ~",  oo_formula_rhs)),
                            data = rdf, weights = weights_oo, design = design_oo,
                            family = quasibinomial(),
                            control = glm.control(maxit = 200))
  wlm_seats_oo   <- svyglm(as.formula(paste("seats ~",   oo_formula_rhs)),
                            data = rdf, weights = weights_oo, design = design_oo,
                            family = quasibinomial(),
                            control = glm.control(maxit = 200))
  wsurv_oo <- coxph(as.formula(paste("Surv(primary_survival, primary) ~", oo_formula_rhs)),
                    data = rdf, weights = weights_oo, robust = TRUE)

  oo_results[[spec_name]] <- list(
    muster = wlm_muster_oo, primary = wlm_primary_oo,
    seats  = wlm_seats_oo,  surv    = wsurv_oo,
    wts = weights_oo, treat = oo_treat, own_var = oo_own
  )

  # Back-compat aliases (suffixed _eb_oo to avoid collision with IPW notebook globals)
  if (spec_name == "ownOther_arak") {
    wlm_muster_oo_eb  <<- wlm_muster_oo
    wlm_primary_oo_eb <<- wlm_primary_oo
    wlm_seats_oo_eb   <<- wlm_seats_oo
    weights_oo_eb     <<- weights_oo
  }

  oo_label_order <- c(oo_treat, oo_own, other_covars, "smHouse", "bigHouse",
                      "friary", "mg_fsnub", "mg_court",
                      "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")
  oo_cov_labels <- unlist(pretty_dict[oo_label_order])

  # Conley SEs (100km Bartlett) for the three svyglm logit columns
  se_muster_oo  <- weighted_conley_se(wlm_muster_oo,  weights_oo)
  se_primary_oo <- weighted_conley_se(wlm_primary_oo, weights_oo)
  se_seats_oo   <- weighted_conley_se(wlm_seats_oo,   weights_oo)

  # capture.output + writeLines avoids the stargazer out= bug with survey objects
  tryCatch({
    tex_lines <- capture.output(
      stargazer(wlm_muster_oo, wlm_primary_oo, wlm_seats_oo, wsurv_oo,
        type = "latex",
        title = paste0("On-site vs. Off-site Monastic Land — EB / Cox PH [", spec_name, "]"),
        label = paste0("tab:eb_ownoff", sfx),
        column.labels = c("Muster", "Primary", "Seats", "Cox PH"),
        se = list(se_muster_oo, se_primary_oo, se_seats_oo, NULL),
        order = paste0("^", oo_label_order, "$"),
        covariate.labels = oo_cov_labels,
        omit = c("Constant", "uplands", "lowlands", "area", "mean_slope"),
        add.lines = list(
          c("Geographic Controls",  "Y", "Y", "Y", "Y"),
          c("EB weights",           "Y", "Y", "Y", "Y"),
          c("Conley SEs (100 km)",  "Y", "Y", "Y", "—"),
          c("Robust SEs (Lin-Wei)", "—", "—", "—", "Y"),
          c("EB treatment", rep(oo_treat, 4))
        ),
        align = TRUE,
        column.sep.width = ".5pt",
        omit.stat = c("aic", "lr", "wald", "logrank"),
        table.placement = "H"
      )
    )
    writeLines(tex_lines, paste0("Output/Tables/EB", sfx, ".tex"))
  }, error = function(e) {
    cat("Stargazer failed for spec", spec_name, ":", e$message, "\n")
  })

  vars_to_plot_oo <- c(oo_treat, oo_own, other_covars,
                       "smHouse", "bigHouse", "friary", "mg_fsnub", "mg_court",
                       "lLStax_pc", "wet_1535", "wet_1536", "lpopC")

  ggsave(paste0("Output/Images/Graphs/eb_logit_primary_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_primary_oo, vars_to_plot_oo, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/eb_logit_muster_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_muster_oo, vars_to_plot_oo, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/eb_logit_seats_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_seats_oo, vars_to_plot_oo, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/eb_cox_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wsurv_oo, vars_to_plot_oo, extract_coefs_coxph),
           pretty_dict, x_label = "Coefficient (Log Hazard Ratio)"),
         width = 10, height = 6, dpi = 300)
}
cat("Section 2 complete — 3 ownOther specs.\n")


===== OwnOther EB spec [ownOther_raw] — treatment lotherLand =====


Warning message:
"Missing values are present in the covariates. See
`?WeightIt::method_ebal` for information on how these are handled."


  ESS: 1143.2 | n: 1755 
                  Summary of weights

- Weight ranges:

      Min                                 Max
all 0.013 |---------------------------| 9.006

- Units with the 5 most extreme weights:
                                 
     1278  1230  1212  1135   987
 all  6.1 6.213 7.047 7.811 9.006

- Weight statistics:

    Coef of Var   MAD Entropy # Zeros
all       0.732 0.434   0.196       0

- Effective Sample Sizes:

             Total
Unweighted 1755.  
Weighted   1143.22



===== OwnOther EB spec [ownOther_sk] — treatment loth_sk =====


Warning message:
"Missing values are present in the covariates. See
`?WeightIt::method_ebal` for information on how these are handled."


  ESS: 1254.4 | n: 1755 
                  Summary of weights

- Weight ranges:

      Min                                 Max
all 0.002 |---------------------------| 6.168

- Units with the 5 most extreme weights:
                                  
       546   528   254   143   119
 all 4.867 5.252 5.686 6.076 6.168

- Weight statistics:

    Coef of Var   MAD Entropy # Zeros
all       0.632 0.401   0.159       0

- Effective Sample Sizes:

             Total
Unweighted 1755.  
Weighted   1254.36



===== OwnOther EB spec [ownOther_arak] — treatment loth_arak =====


Warning message:
"Missing values are present in the covariates. See
`?WeightIt::method_ebal` for information on how these are handled."


  ESS: 1184.9 | n: 1755 
                  Summary of weights

- Weight ranges:

      Min                                  Max
all 0.009 |---------------------------| 14.528

- Units with the 5 most extreme weights:
                                   
      1415  1402  1230  1160    178
 all 5.292 5.624 7.316 7.887 14.528

- Weight statistics:

    Coef of Var   MAD Entropy # Zeros
all       0.694 0.383    0.16       0

- Effective Sample Sizes:

             Total
Unweighted 1755.  
Weighted   1184.88


Section 2 complete — 3 ownOther specs.


## Section 3: Conley Standard Errors — EB Robustness

Spatial-HAC standard errors for the EB (svyglm) outcome models. Because
`conleyreg` does not accept observation weights, we implement the weighted
sandwich directly: for each model we compute the EB-weighted residuals
\(r_i = w_i (y_i - \mu_i)\), the working-weight bread \(B = (X' \mathrm{diag}(w \mu (1-\mu)) X)^{-1}\),
and the spatial meat \(M = X' \mathrm{diag}(r) \, \Omega \, \mathrm{diag}(r) X\)
where \(\Omega_{ij} = \max(1 - d_{ij}/c, 0)\) (Bartlett kernel) and \(d_{ij}\) is
the centroid-to-centroid km distance in EPSG:27700.
SEs reported at cutoffs of 50, 100, and 150 km.

In [6]:
eb_conley_cutoffs <- c(50, 100, 150)

# --- Helper: write one Conley-SE table (muster / primary / seats) --------
write_eb_conley_table <- function(m_muster, m_primary, m_seats, wts,
                                  cutoff_km, title, label_suffix, out_path,
                                  covariate_labels, cov_order) {
  se_muster  <- weighted_conley_se(m_muster,  wts, cutoff_km)
  se_primary <- weighted_conley_se(m_primary, wts, cutoff_km)
  se_seats   <- weighted_conley_se(m_seats,   wts, cutoff_km)
  stargazer(
    m_muster, m_primary, m_seats,
    type             = "latex",
    se               = list(se_muster, se_primary, se_seats),
    title            = title,
    label            = paste0("tab:conley_eb", label_suffix, "_", cutoff_km, "km"),
    column.labels    = c("Muster", "Primary", "Seats"),
    covariate.labels = covariate_labels,
    order            = paste0("^", cov_order, "$"),
    omit             = c("Constant", "uplands", "lowlands", "area", "mean_slope"),
    omit.stat        = c("aic", "lr", "wald", "logrank"),
    add.lines        = list(
      c("Conley cutoff (km)", rep(as.character(cutoff_km), 3)),
      c("Kernel",             rep("Bartlett",             3)),
      c("EB weights",         rep("Y",                    3))
    ),
    align            = TRUE,
    column.sep.width = ".5pt",
    table.placement  = "H",
    out              = out_path
  )
}

# --- (a) Main specs (6 total) --------------------------------------------
for (spec_name in names(main_results)) {
  res  <- main_results[[spec_name]]
  sfx  <- main_specs[[spec_name]]$suffix
  tr   <- res$tr
  monc <- res$monc

  cov_order  <- c(tr, monc, "smHouse", "bigHouse", "friary",
                  "mg_fsnub", "mg_court",
                  "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")
  cov_labels <- unlist(pretty_dict[cov_order])

  for (k in eb_conley_cutoffs) {
    write_eb_conley_table(
      m_muster         = res$muster,
      m_primary        = res$primary,
      m_seats          = res$seats,
      wts              = res$wts,
      cutoff_km        = k,
      title            = paste0("EB Logits — Conley SEs [", spec_name, "], cutoff = ", k, " km"),
      label_suffix     = sfx,
      out_path         = paste0("Output/Tables/conley_eb", sfx, "_", k, "km.tex"),
      covariate_labels = cov_labels,
      cov_order        = cov_order
    )
  }
  cat("Wrote Conley EB tables for spec:", spec_name, "\n")
}

# --- (b) OwnOther specs (3 total) ----------------------------------------
for (spec_name in names(oo_results)) {
  res     <- oo_results[[spec_name]]
  sfx     <- oo_specs[[spec_name]]$suffix
  oo_tr   <- res$treat
  oo_own  <- res$own_var
  oo_oths <- oo_specs[[spec_name]]$other_covars

  cov_order  <- c(oo_tr, oo_own, oo_oths, "smHouse", "bigHouse", "friary",
                  "mg_fsnub", "mg_court",
                  "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")
  cov_labels <- unlist(pretty_dict[cov_order])

  for (k in eb_conley_cutoffs) {
    write_eb_conley_table(
      m_muster         = res$muster,
      m_primary        = res$primary,
      m_seats          = res$seats,
      wts              = res$wts,
      cutoff_km        = k,
      title            = paste0("EB OwnOther — Conley SEs [", spec_name, "], cutoff = ", k, " km"),
      label_suffix     = sfx,
      out_path         = paste0("Output/Tables/conley_eb", sfx, "_", k, "km.tex"),
      covariate_labels = cov_labels,
      cov_order        = cov_order
    )
  }
  cat("Wrote Conley EB tables for spec:", spec_name, "\n")
}

cat("\nEB Conley SE tables written to Output/Tables/conley_eb_*.tex\n")


% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 13:24:16
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{EB Logits — Conley SEs [total_raw], cutoff = 50 km} 
  \label{tab:conley_eb_total_raw_50km} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{paste("muster \textasciitilde", rhs)} & \multicolumn{1}{c}{paste("primary \textasciitilde", rhs)} & \multicolumn{1}{c}{paste("seats \textasciitilde", rhs)} \\ 
 & \multicolumn{1}{c}{Muster} & \multicolumn{1}{c}{Primary} & \multicolumn{1}{c}{Seats} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)}\\ 
\hline \\[-1.8ex] 
 ln(Total Monastic Land Owned + 1) & 0.193 & 0.585 & 32,385,272,712,688.000 \\ 
  & (0.197) &


% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 13:24:44
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{EB OwnOther — Conley SEs [ownOther_raw], cutoff = 50 km} 
  \label{tab:conley_eb_ownOther_raw_50km} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{paste("muster \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{paste("primary \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{paste("seats \textasciitilde", oo\_formula\_rhs)} \\ 
 & \multicolumn{1}{c}{Muster} & \multicolumn{1}{c}{Primary} & \multicolumn{1}{c}{Seats} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)}\\ 
\hline \\[-1.8ex] 
 ln(Off-site Monastic Land + 1) & 0.201 & 0.59


EB Conley SE tables written to Output/Tables/conley_eb_*.tex
